In [1]:
import pandas as pd

In [2]:
nt = pd.read_excel('./report.xlsx', sheet_name='tags_report')
np = pd.read_excel('./report.xlsx', sheet_name='projects_report')
nc = pd.read_excel('./report.xlsx', sheet_name='clients_report')

In [3]:
main_excel_path = './data'
df = pd.read_excel(f'{main_excel_path}/New tags and projects list_Toggl.xlsx', sheet_name='1.2-User-specific tags',
                   index_col='Activity type')
tags_format = pd.read_excel(f'{main_excel_path}/New tags and projects list_Toggl.xlsx', sheet_name='Tags - Format')
projects = pd.read_excel(f'{main_excel_path}/New tags and projects list_Toggl.xlsx', sheet_name='Projects')
clients = pd.read_excel(f'{main_excel_path}/New tags and projects list_Toggl.xlsx', sheet_name='Client Name')

In [4]:
print(f'nt cols: {nt.columns}, np cols: {np.columns}, nc cols: {nc.columns}')

nt cols: Index(['id', 'workspace_id', 'name', 'at', 'creator_id', 'user'], dtype='object'), np cols: Index(['id', 'workspace_id', 'client_id', 'name', 'is_private', 'active', 'at',
       'created_at', 'server_deleted_at', 'color', 'billable', 'template',
       'auto_estimates', 'estimated_hours', 'estimated_seconds', 'rate',
       'rate_last_updated', 'currency', 'recurring', 'template_id',
       'recurring_parameters', 'fixed_fee', 'actual_hours', 'actual_seconds',
       'total_count', 'client_name', 'can_track_time', 'start_date', 'status',
       'wid', 'cid', 'pinned', 'user'],
      dtype='object'), nc cols: Index(['id', 'wid', 'archived', 'name', 'at', 'creator_id', 'total_count',
       'user'],
      dtype='object')


In [5]:
print(f'User specific tags - df cols: {df.columns}, tags_format cols: {tags_format.columns}, projects cols: {projects.columns}, clients cols: {clients.columns}')

User specific tags - df cols: Index(['ANAS', 'NIAL', 'SMVE', 'AMAN', 'DIDA', 'AIVI', 'NIRO', 'VAEL', 'BODM',
       'KAAN', 'KURO', 'CHAL', 'BADA', 'LAOK', 'PRAR', 'TODA',
       'AT definition'],
      dtype='object'), tags_format cols: Index(['Tags - Format', 'Update Date', 'Comment'], dtype='object'), projects cols: Index(['Projects', 'Update Date', 'Comment'], dtype='object'), clients cols: Index(['Client Name', 'Update Date', 'Comment'], dtype='object')


In [16]:
import pandas as pd
import re

def normalize_tag(tag: str) -> str:
    if pd.isna(tag):
        return tag
    tag = tag.strip().lower()
    tag = re.sub(r'[-:]', '', tag)  # убираем - и :
    tag = re.sub(r'\s+', ' ', tag)  # нормализация пробелов
    return tag


def build_tags_diff(df_expected: pd.DataFrame, nt: pd.DataFrame) -> pd.DataFrame:
    
    # --- 0. Expected -> long
    expected_long = (
        df_expected
        .reset_index()
        .melt(id_vars=df_expected.index.name or 'index',
              var_name='user',
              value_name='expected')
        .rename(columns={df_expected.index.name or 'index': 'tag'})
    )
    
    # --- фильтр expected
    expected_long = expected_long[expected_long['expected'] == 1][['user', 'tag']]
    
    # --- добавляем format в expected
    format_tags = tags_format[['Tags - Format']].rename(columns={'Tags - Format': 'tag'})
    
    users = expected_long[['user']].drop_duplicates()
    
    format_expanded_expected = users.merge(format_tags, how='cross')
    
    expected_long = pd.concat([expected_long, format_expanded_expected], ignore_index=True).drop_duplicates()
    
    # --- 1. Actual
    # --- 1. Actual (основные теги)
    actual_main = nt[['user', 'name']].rename(columns={'name': 'tag'})
    
    # --- 1.1 Format теги (справочник)
    format_tags = tags_format[['Tags - Format']].rename(columns={'Tags - Format': 'tag'})
    
    # --- 1.2 Получаем всех пользователей
    users = actual_main[['user']].drop_duplicates()
    
    # --- 1.3 Декартово произведение: каждый user получает все format-теги
    format_expanded = users.merge(format_tags, how='cross')
    
    # --- 1.4 Объединение
    actual = pd.concat([actual_main, format_expanded], ignore_index=True).drop_duplicates()
    
    # --- 3. Нормализация
    expected_long['norm'] = expected_long['tag'].apply(normalize_tag)
    actual['norm'] = actual['tag'].apply(normalize_tag)
    
    # --- 4. REPLACE (ключевая часть)
    replace_df = (
        actual
        .merge(expected_long, on=['user', 'norm'], suffixes=('_actual', '_expected'))
    )
    
    replace_df = replace_df[
        replace_df['tag_actual'] != replace_df['tag_expected']
    ][['user', 'tag_actual', 'tag_expected']]
    
    replace_df = replace_df.rename(columns={
        'tag_actual': 'name',
        'tag_expected': 'right_variant'
    })
    replace_df['doing'] = 'replace'
    
    # --- 5. Убираем уже учтенные replace из выборок
    replace_keys = set(zip(replace_df['user'], replace_df['name']))
    
    actual_filtered = actual[
        ~actual.apply(lambda x: (x['user'], x['tag']) in replace_keys, axis=1)
    ]
    
    # --- 6. ADD
    add_df = (
        expected_long
        .merge(actual_filtered, on=['user', 'norm', 'tag'], how='left', indicator=True)
    )
    
    add_df = add_df[add_df['_merge'] == 'left_only'][['user', 'tag']]
    add_df = add_df.rename(columns={'tag': 'name'})
    add_df['doing'] = 'add'
    add_df['right_variant'] = add_df['name']
    
    # --- 7. DELETE
    delete_df = (
        actual_filtered
        .merge(expected_long, on=['user', 'norm', 'tag'], how='left', indicator=True)
    )
    
    delete_df = delete_df[delete_df['_merge'] == 'left_only'][['user', 'tag']]
    delete_df = delete_df.rename(columns={'tag': 'name'})
    delete_df['doing'] = 'delete'
    delete_df['right_variant'] = delete_df['name']
    
    # --- 8. Объединение
    result = pd.concat([add_df, delete_df, replace_df], ignore_index=True)
    
    return result[['user', 'name', 'doing', 'right_variant']]

In [19]:
tags_diff = build_tags_diff(df, nt)
# tags_diff.loc[tags_diff.user == 'CHAL', 'name'].unique()
tags_diff

array(['CAT: 1 |  Исследование разрозненных источников данных в клиентских информационных системах',
       'CAT: 2 | Планирование сбора данных в хранилище (наладка ETL совместно с системными инженерами)',
       'CAT: 3 | Оценка чистоты данных, участие в разработке правил такой оценки',
       'CAT: 4 | Разработка и доработка BI-отчетов и дашбордов',
       'CAT: 5 | Разработка рабочих прототипов программных продуктов (Minimum Viable Product, далее - MVP)',
       'CAT: 6 | Поддержка отдела информационных технологий Заказчика в рамках преобразования MVP',
       'CAT: 7 | Услуги по аналитике данных в рамках консалтинговых проектов Заказчика',
       'CAT: 8 | Дополнителные выплаты'], dtype=object)

In [31]:
tags_diff['name'].unique()

array(['CAT: 1 |  Исследование разрозненных источников данных в клиентских информационных системах',
       'CAT: 2 | Планирование сбора данных в хранилище (наладка ETL совместно с системными инженерами)',
       'CAT: 3 | Оценка чистоты данных, участие в разработке правил такой оценки',
       'CAT: 4 | Разработка и доработка BI-отчетов и дашбордов',
       'CAT: 5 | Разработка рабочих прототипов программных продуктов (Minimum Viable Product, далее - MVP)',
       'CAT: 6 | Поддержка отдела информационных технологий Заказчика в рамках преобразования MVP',
       'CAT: 7 | Услуги по аналитике данных в рамках консалтинговых проектов Заказчика',
       'CAT: 8 | Дополнителные выплаты', 'pro', 'Dep: Consulting'],
      dtype=object)

In [34]:
exception_list = ['CAT: 1 |  Исследование разрозненных источников данных в клиентских информационных системах',
                   'CAT: 2 | Планирование сбора данных в хранилище (наладка ETL совместно с системными инженерами)',
                   'CAT: 3 | Оценка чистоты данных, участие в разработке правил такой оценки',
                   'CAT: 4 | Разработка и доработка BI-отчетов и дашбордов',
                   'CAT: 5 | Разработка рабочих прототипов программных продуктов (Minimum Viable Product, далее - MVP)',
                   'CAT: 6 | Поддержка отдела информационных технологий Заказчика в рамках преобразования MVP',
                   'CAT: 7 | Услуги по аналитике данных в рамках консалтинговых проектов Заказчика',
                   'CAT: 8 | Дополнителные выплаты',
                 
                 ]

tags_diff.drop(tags_diff[tags_diff.name.isin(exception_list)].index, axis=0, inplace=True)
tags_diff

,user,name,doing,right_variant
8,AMAN,pro,delete,pro
9,SMVE,Dep: Consulting,delete,Dep: Consulting


In [8]:
tags_format

,Tags - Format,Update Date,Comment
0,Format: Individual work,01.07.2025,0
1,Format: Internal meeting,01.07.2025,0
2,Format: Meeting with Vendor,01.07.2025,0
3,Format: Meeting with Lab,01.07.2025,0
4,Format: Meeting with Hospital,09.12.2025,0


In [9]:
projects

,Projects,Update Date,Comment
0,Abbott: Sorters BI,01.07.2025,0
1,Alrijne: Assessment,30.09.2025,0
2,Alrijne: General,09.12.2025,0
3,Alrijne: Hospital Lab-Design,09.12.2025,0
4,Alrijne: Lab-Avatar,01.07.2025,0
...,...,...,...
63,StreekLab: Lab-Avatar,01.07.2025,0
64,StreekLab: Lab-Calculator,01.07.2025,0
65,StreekLab: Lab-Data,01.07.2025,0
66,StreekLab: Tender Support,01.07.2025,0


In [10]:
clients

,Client Name,Update Date,Comment
0,Abbott,01.07.2025,0
1,Alrijne,01.07.2025,0
2,BC 6S,09.12.2025,0
3,BC Cork&Kerry,01.07.2025,0
4,BC Europe&UK,09.12.2025,0
5,BC Fr Bordeaux,09.12.2025,0
6,BC GP,18.03.2026,0
7,BC MSC,01.07.2025,0
8,BC SWLP,01.07.2025,0
9,BC WoE,01.07.2025,0


In [11]:
import pandas as pd
import re

def normalize_tag(tag: str) -> str:
    if pd.isna(tag):
        return tag
    tag = tag.strip().lower()
    tag = re.sub(r'[-:]', '', tag)
    tag = re.sub(r'\s+', ' ', tag)
    return tag


def build_diff_entities(df_expected: pd.DataFrame, actual_df: pd.DataFrame, entity: str) -> pd.DataFrame:
    """
    entity: 'projects' или 'clients'
    """

    # --- 1. Определяем колонки
    if entity == 'projects':
        expected_col = 'Projects'
        actual_col = 'name'
    elif entity == 'clients':
        expected_col = 'Client Name'
        actual_col = 'name'
    else:
        raise ValueError("entity must be 'projects' or 'clients'")

    # --- 2. Expected
    expected = (
        df_expected[[expected_col]]
        .rename(columns={expected_col: 'name'})
        .dropna()
        .drop_duplicates()
    )

    expected['norm'] = expected['name'].apply(normalize_tag)

    # --- 3. Actual
    actual = (
        actual_df[['user', actual_col]]
        .rename(columns={actual_col: 'name'})
        .dropna()
        .drop_duplicates()
    )

    actual['norm'] = actual['name'].apply(normalize_tag)

    # --- 4. REPLACE
    replace_df = (
        actual
        .merge(expected, on='norm', suffixes=('_actual', '_expected'))
    )

    replace_df = replace_df[
        replace_df['name_actual'] != replace_df['name_expected']
    ][['user', 'name_actual', 'name_expected']]

    replace_df = replace_df.rename(columns={
        'name_actual': 'name',
        'name_expected': 'right_variant'
    })

    replace_df['doing'] = 'replace'

    # --- 5. Убираем replace из дальнейших расчетов (по norm!)
    replace_keys = set(zip(replace_df['user'], replace_df['name'].apply(normalize_tag)))

    actual_filtered = actual[
        ~actual.apply(lambda x: (x['user'], x['norm']) in replace_keys, axis=1)
    ]

    # --- 6. ADD (expected нет в actual)
    add_df = (
        expected
        .merge(actual_filtered[['norm']].drop_duplicates(), on='norm', how='left', indicator=True)
    )

    add_df = add_df[add_df['_merge'] == 'left_only'][['name']]
    add_df['doing'] = 'add'
    add_df['right_variant'] = add_df['name']
    add_df['user'] = None

    # --- 7. DELETE (actual нет в expected)
    delete_df = (
        actual_filtered
        .merge(expected[['norm']], on='norm', how='left', indicator=True)
    )

    delete_df = delete_df[delete_df['_merge'] == 'left_only'][['user', 'name']]
    delete_df['doing'] = 'delete'
    delete_df['right_variant'] = delete_df['name']

    # --- 8. Объединение
    result = pd.concat([add_df, delete_df, replace_df], ignore_index=True)

    return result[['user', 'name', 'doing', 'right_variant']]

In [13]:
projects_diff = build_diff_entities(projects, np, 'projects')
projects_diff

,user,name,doing,right_variant


In [14]:
clients_diff  = build_diff_entities(clients, nc, 'clients')
clients_diff

,user,name,doing,right_variant


In [36]:
import pandas as pd

def export_diffs_to_excel(tags_diff: pd.DataFrame,
                          projects_diff: pd.DataFrame,
                          clients_diff: pd.DataFrame,
                          filename: str = './data/diff_tpc.xlsx'):
    
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        tags_diff.to_excel(writer, sheet_name='tags', index=False)
        projects_diff.to_excel(writer, sheet_name='projects', index=False)
        clients_diff.to_excel(writer, sheet_name='clients', index=False)

    print(f"Saved to {filename}")

In [37]:
export_diffs_to_excel(
    tags_diff=tags_diff,
    projects_diff=projects_diff,
    clients_diff=clients_diff,
    filename='./data/diff_tpc.xlsx'
)

Saved to ./data/diff_tpc.xlsx
